In [0]:
-- Team - Shot Distance For vs Against
-- Shots and goals by distance from the net, for and against.
-- Goal and shot totals come from play-by-play and leave out shootouts.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    'MTL' AS team,       -- team abbreviation; CAST(NULL AS STRING) = every team
    20252026 AS season,  -- e.g. 20242025; CAST(NULL AS INT) = every season
    2 AS game_type       -- 2 = regular season, 3 = playoffs
),
team_games AS (                          -- one row per team per finished game
  SELECT g.game_id, g.season, g.game_date, g.date_key, g.last_period_type,
         t.team_id, t.abbrev AS team, o.abbrev AS opponent,
         t.team_id = g.home_team_id AS is_home, g.home_team_id,
         COALESCE(t.abbrev = pr.team, FALSE) AS is_selected,
         IF(t.team_id = g.home_team_id, g.home_final_score, g.away_final_score) AS final_gf,
         IF(t.team_id = g.home_team_id, g.away_final_score, g.home_final_score) AS final_ga,
         CASE WHEN IF(t.team_id = g.home_team_id, g.home_final_score > g.away_final_score,
                                                  g.away_final_score > g.home_final_score) THEN 'W'
              WHEN g.game_type = 2 AND g.last_period_type IN ('OT', 'SO') THEN 'OTL'
              ELSE 'L' END AS result
  FROM nhl.gold.dim_game g
  JOIN nhl.gold.dim_team t ON t.team_id IN (g.home_team_id, g.away_team_id)
  JOIN nhl.gold.dim_team o ON o.team_id = IF(t.team_id = g.home_team_id, g.away_team_id, g.home_team_id)
  CROSS JOIN params pr
  WHERE g.is_final
    AND g.game_type = pr.game_type
    AND (pr.season IS NULL OR g.season = pr.season)
    AND (pr.team IS NULL OR t.abbrev = pr.team)
),
team_events AS (                         -- plays in those games, from that team's side
  SELECT tg.team, tg.team_id, tg.opponent, tg.season, tg.game_date, tg.is_home, tg.result,
         IF(f.event_owner_team_id = tg.team_id, 'for', 'against') AS side,
         IF(f.event_owner_team_id = tg.home_team_id, f.situation_code, reverse(f.situation_code)) AS owner_situation,
         CASE WHEN substr(owner_situation, 1, 1) = '0' THEN 'EN'      -- opponent net empty
              WHEN substr(owner_situation, 3, 1) > substr(owner_situation, 2, 1)
               AND substr(owner_situation, 4, 1) = '1' THEN 'PP'      -- more skaters, own goalie in
              WHEN substr(owner_situation, 3, 1) < substr(owner_situation, 2, 1) THEN 'SH'
              ELSE 'EV' END AS strength,
         br.team_id AS blocker_team_id,
         f.*
  FROM nhl.gold.fact_event f
  JOIN team_games tg ON f.game_id = tg.game_id
  LEFT JOIN nhl.silver.game_rosters br                -- blocked shots belong to the shooter's team,
         ON br.game_id = f.game_id AND br.player_id = f.blocking_player_id   -- so credit blocks by roster
  WHERE COALESCE(f.period_type, '') <> 'SO'    -- shootout attempts aren't stats
)
SELECT CASE WHEN SQRT(POW(89 - ABS(te.x_coord), 2) + POW(te.y_coord, 2)) < 15 THEN '1) 0-14 ft' WHEN SQRT(POW(89 - ABS(te.x_coord), 2) + POW(te.y_coord, 2)) < 30 THEN '2) 15-29 ft'
              WHEN SQRT(POW(89 - ABS(te.x_coord), 2) + POW(te.y_coord, 2)) < 45 THEN '3) 30-44 ft' ELSE '4) 45+ ft' END AS distance,
       COUNT_IF(side = 'for')                                                   AS shots_for,
       COUNT_IF(side = 'for' AND type_desc_key = 'goal')                        AS goals_for,
       ROUND(100.0 * COUNT_IF(side = 'for' AND type_desc_key = 'goal')
                   / NULLIF(COUNT_IF(side = 'for'), 0), 1)                      AS shooting_pct_for,
       COUNT_IF(side = 'against')                                               AS shots_against,
       COUNT_IF(side = 'against' AND type_desc_key = 'goal')                    AS goals_against,
       ROUND(100.0 * COUNT_IF(side = 'against' AND type_desc_key = 'goal')
                   / NULLIF(COUNT_IF(side = 'against'), 0), 1)                  AS shooting_pct_against
FROM team_events te
WHERE type_desc_key IN ('goal', 'shot-on-goal') AND zone_code = 'O'   -- zone is from the shooter's side
GROUP BY 1
ORDER BY distance;